# Experiment 6: Bagging, Boosting, and Stacked Ensemble Models

Wisconsin Diagnostic Breast Cancer dataset, loaded via `sklearn.datasets.load_breast_cancer` (569 samples, 30 features, malignant/benign).

Compares Bagging (Decision Tree base), Boosting (AdaBoost + Gradient Boosting), and a Stacked Ensemble (SVM + Naive Bayes + Decision Tree -> Logistic Regression meta-learner).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier,
                               StackingClassifier)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix)

sns.set_style("whitegrid")
rng = 42

## 1. Load and Preprocess Dataset

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="diagnosis")  # 0=malignant, 1=benign (already encoded)
target_names = data.target_names

print("Shape:", X.shape)
print("Missing values:", X.isnull().sum().sum())
y.map({0: "malignant", 1: "benign"}).value_counts()

## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x=y.map({0: "malignant", 1: "benign"}))
plt.title("Class Distribution -- Breast Cancer Wisconsin (Diagnostic)")
plt.show()

In [ ]:
top_corr_feats = X.corrwith(y).abs().sort_values(ascending=False).head(12).index.tolist()
plt.figure(figsize=(10,8))
sns.heatmap(pd.concat([X[top_corr_feats], y], axis=1).corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap -- Top 12 Features Correlated with Diagnosis")
plt.show()

## 3. Train-Test Split (80:20, Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rng)
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
def evaluate(model, Xtr, ytr, Xte, yte):
    t0 = time.time()
    model.fit(Xtr, ytr)
    train_time = time.time() - t0
    t0 = time.time()
    pred = model.predict(Xte)
    pred_time = time.time() - t0
    proba = model.predict_proba(Xte)[:, 1]
    metrics = dict(
        Accuracy=accuracy_score(yte, pred),
        Precision=precision_score(yte, pred),
        Recall=recall_score(yte, pred),
        F1=f1_score(yte, pred),
        ROC_AUC=roc_auc_score(yte, proba),
        TrainTime=train_time,
        PredictTime=pred_time,
    )
    return metrics, pred, proba

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=rng)

## 4. Bagging Classifier (Base Estimator: Decision Tree)

In [ ]:
bag_baseline = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=rng), random_state=rng)
bag_baseline_metrics, _, _ = evaluate(bag_baseline, X_train, y_train, X_test, y_test)
bag_baseline_metrics

In [ ]:
bag_param_grid = {
    "n_estimators": [10, 50, 100, 200],
    "max_samples": [0.5, 0.7, 1.0],
    "max_features": [0.5, 0.7, 1.0],
}
bag_grid = GridSearchCV(BaggingClassifier(estimator=DecisionTreeClassifier(random_state=rng), random_state=rng),
                         param_grid=bag_param_grid, cv=skf, scoring="accuracy", n_jobs=-1)
bag_grid.fit(X_train, y_train)
print("Best params:", bag_grid.best_params_)
print("Best CV accuracy:", round(bag_grid.best_score_, 4))

In [ ]:
best_bag = bag_grid.best_estimator_
best_bag_metrics, best_bag_pred, best_bag_proba = evaluate(best_bag, X_train, y_train, X_test, y_test)
best_bag_metrics

## 5. Boosting: AdaBoost and Gradient Boosting

In [ ]:
ada_baseline = AdaBoostClassifier(random_state=rng)
ada_baseline_metrics, _, _ = evaluate(ada_baseline, X_train, y_train, X_test, y_test)

gb_baseline = GradientBoostingClassifier(random_state=rng)
gb_baseline_metrics, _, _ = evaluate(gb_baseline, X_train, y_train, X_test, y_test)

print("AdaBoost baseline:", ada_baseline_metrics)
print("Gradient Boosting baseline:", gb_baseline_metrics)

In [ ]:
boost_param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.5, 1.0],
    "max_depth": [1, 2, 3],
}
gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=rng), param_grid=boost_param_grid,
                        cv=skf, scoring="accuracy", n_jobs=-1)
gb_grid.fit(X_train, y_train)
print("Best params:", gb_grid.best_params_)
print("Best CV accuracy:", round(gb_grid.best_score_, 4))

In [ ]:
best_gb = gb_grid.best_estimator_
best_gb_metrics, best_gb_pred, best_gb_proba = evaluate(best_gb, X_train, y_train, X_test, y_test)
best_gb_metrics

In [ ]:
ada_param_grid = {"n_estimators": [50, 100, 200], "learning_rate": [0.01, 0.1, 0.5, 1.0]}
ada_grid = GridSearchCV(AdaBoostClassifier(random_state=rng), param_grid=ada_param_grid,
                         cv=skf, scoring="accuracy", n_jobs=-1)
ada_grid.fit(X_train, y_train)
print("Best params:", ada_grid.best_params_)
print("Best CV accuracy:", round(ada_grid.best_score_, 4))

## 6. Stacked Ensemble (SVM + Naive Bayes + Decision Tree -> Logistic Regression)

In [ ]:
base_learners = [
    ("svm", SVC(probability=True, random_state=rng)),
    ("nb", GaussianNB()),
    ("dt", DecisionTreeClassifier(random_state=rng)),
]
stack_baseline = StackingClassifier(estimators=base_learners,
                                     final_estimator=LogisticRegression(max_iter=5000, random_state=rng),
                                     cv=skf)
stack_baseline_metrics, stack_pred, stack_proba = evaluate(stack_baseline, X_train, y_train, X_test, y_test)
stack_baseline_metrics

In [ ]:
stack_cv_scores = cross_val_score(stack_baseline, X_train, y_train, cv=skf, scoring="accuracy")
stack_cv_f1 = cross_val_score(stack_baseline, X_train, y_train, cv=skf, scoring="f1")
print("Avg CV accuracy:", round(stack_cv_scores.mean(), 4))
print("Avg CV F1:", round(stack_cv_f1.mean(), 4))

In [ ]:
# Alternative base-model combination (drop Naive Bayes) to test heterogeneity value
base_learners_alt = [
    ("svm", SVC(probability=True, random_state=rng)),
    ("dt", DecisionTreeClassifier(random_state=rng)),
]
stack_alt = StackingClassifier(estimators=base_learners_alt,
                                final_estimator=LogisticRegression(max_iter=5000, random_state=rng),
                                cv=skf)
stack_alt_cv_scores = cross_val_score(stack_alt, X_train, y_train, cv=skf, scoring="accuracy")
print("Alt (SVM+DT only) avg CV accuracy:", round(stack_alt_cv_scores.mean(), 4))

## 7. 5-Fold Cross-Validation Comparison Across All Three Ensembles

In [ ]:
bag_cv_final = cross_val_score(BaggingClassifier(estimator=DecisionTreeClassifier(random_state=rng),
                                                  **bag_grid.best_params_, random_state=rng),
                                X, y, cv=skf, scoring="accuracy")
boost_cv_final = cross_val_score(GradientBoostingClassifier(**gb_grid.best_params_, random_state=rng),
                                  X, y, cv=skf, scoring="accuracy")
stack_cv_final = cross_val_score(stack_baseline, X, y, cv=skf, scoring="accuracy")

print("Bagging:", bag_cv_final.round(4), " avg:", round(bag_cv_final.mean(),4))
print("Boosting:", boost_cv_final.round(4), " avg:", round(boost_cv_final.mean(),4))
print("Stacking:", stack_cv_final.round(4), " avg:", round(stack_cv_final.mean(),4))

In [ ]:
plt.figure(figsize=(7,4))
folds = list(range(1,6))
plt.plot(folds, bag_cv_final, "o-", label="Bagging")
plt.plot(folds, boost_cv_final, "o-", label="Boosting (Gradient Boosting)")
plt.plot(folds, stack_cv_final, "o-", label="Stacking")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("5-Fold Cross-Validation Accuracy -- Ensemble Comparison")
plt.legend()
plt.show()

## 8. Bias-Variance Illustration: n_estimators Curves

In [ ]:
n_est_values = [1, 5, 10, 25, 50, 100, 200]
bag_curve, boost_curve = {}, {}
for n in n_est_values:
    b = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=rng), n_estimators=n, random_state=rng)
    b.fit(X_train, y_train)
    bag_curve[n] = (accuracy_score(y_train, b.predict(X_train)), accuracy_score(y_test, b.predict(X_test)))

    g = GradientBoostingClassifier(n_estimators=n if n > 0 else 1, random_state=rng)
    g.fit(X_train, y_train)
    boost_curve[n] = (accuracy_score(y_train, g.predict(X_train)), accuracy_score(y_test, g.predict(X_test)))

fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
axes[0].plot(n_est_values, [bag_curve[n][0] for n in n_est_values], "o-", label="Train Accuracy")
axes[0].plot(n_est_values, [bag_curve[n][1] for n in n_est_values], "o-", label="Test Accuracy")
axes[0].set_xlabel("n_estimators")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Bagging: Train/Test Accuracy vs. n_estimators")
axes[0].legend()

axes[1].plot(n_est_values, [boost_curve[n][0] for n in n_est_values], "o-", label="Train Accuracy")
axes[1].plot(n_est_values, [boost_curve[n][1] for n in n_est_values], "o-", label="Test Accuracy")
axes[1].set_xlabel("n_estimators")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Boosting: Train/Test Accuracy vs. n_estimators")
axes[1].legend()
plt.show()

## Confusion Matrices, ROC Curves, and Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13,4))
for ax, (name, pred) in zip(axes, [("Bagging", best_bag_pred), ("Boosting", best_gb_pred), ("Stacking", stack_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["malignant","benign"], yticklabels=["malignant","benign"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6.5,5.5))
for name, proba in [("Bagging", best_bag_proba), ("Boosting", best_gb_proba), ("Stacking", stack_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
plt.plot([0,1],[0,1],"k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves -- Bagging vs. Boosting vs. Stacking")
plt.legend()
plt.show()

In [ ]:
compare_df = pd.DataFrame({
    "Bagging": best_bag_metrics,
    "Boosting": best_gb_metrics,
    "Stacking": stack_baseline_metrics,
}).T[["Accuracy", "Precision", "Recall", "F1"]]

compare_df.plot(kind="bar", figsize=(8,5))
plt.title("Ensemble Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()
compare_df

## Summary

See the accompanying report (`Experiment7.pdf`) for the full hyperparameter tuning tables, observation-question answers, and discussion grounded in these results.